リンク：https://docs.databricks.com/aws/ja/generative-ai/agent-framework/stateful-agents

# Mosaic AI Agent Framework: Lakebaseをストアとして使用した長期記憶を持つステートフルエージェントの作成とデプロイ
このノートブックは、Mosaic AI Agent FrameworkとLakebaseをエージェントのメモリストアとして使用して、ユーザーの好みを保存・取得するステートフルエージェントを構築する方法を示します。

このノートブックでは、以下を行います:
1. Lakebaseを使用した長期記憶エージェントグラフの作成(ストア内のセマンティック検索を介してユーザーの好みを保存・呼び出し)
2. LangGraphエージェントを`ResponsesAgent`インターフェースでラップし、Databricks機能との互換性を確保
3. エージェントの動作をローカルでテスト
4. モデルをUnity Catalogに登録し、ログを記録してエージェントをデプロイし、アプリやPlaygroundで使用

## 前提条件
- Lakebaseインスタンスが準備され、実行されていること。Databricksドキュメントを参照してください([AWS](https://docs.databricks.com/aws/en/oltp/create/) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/oltp/create/))。
- Lakebaseインスタンスは、SQL Warehouses -> Lakebase Postgres -> Create database instanceから作成できます。このノートブックに記入するために、Lakebaseの「Connection details」セクションから値を取得する必要があります。
- このノートブック全体の「TODO」をすべて完了してください

### 依存関係のインストール

In [0]:
%pip install -U -qqqq uv databricks-agents mlflow-skinny[databricks] databricks-langchain[memory]
dbutils.library.restartPython()

## 初回セットアップのみ: Lakebaseインスタンス用のストアテーブルを設定

In [0]:
from databricks.sdk import WorkspaceClient
from databricks_langchain import DatabricksStore

# TODO: Lakebaseの設定値を入力してください
LAKEBASE_INSTANCE_NAME = "lakebase-name"

store = DatabricksStore(instance_name=LAKEBASE_INSTANCE_NAME)
store.setup()

# エージェントをコードで定義

## エージェントコードをファイルagent.pyに書き込み
以下の単一セルでエージェントコードを定義します。これにより、`%%writefile`マジックコマンドを使用してエージェントコードをローカルPythonファイルに書き込み、後続のログ記録とデプロイに使用できます。

## ResponsesAgentインターフェースを使用してLangGraphエージェントをラップ
Databricks AI機能との互換性のために、`LangGraphResponsesAgent`クラスは`ResponsesAgent`インターフェースを実装してLangGraphエージェントをラップします。

Databricksは、オープンソース標準を使用してマルチターン会話エージェントの作成を簡素化するため、`ResponsesAgent`の使用を推奨しています。MLflowの[ResponsesAgentドキュメント](https://www.mlflow.org/docs/latest/llms/responses-agent-intro/)を参照してください。

In [0]:
%%writefile agent.py
import json
import logging
import os
from typing import Annotated, Any, Generator, Optional, Sequence, TypedDict

import mlflow
from databricks_langchain import (
    ChatDatabricks,
    DatabricksStore,
    UCFunctionToolkit,
)
from langchain_core.messages import (
    AIMessage,
    AIMessageChunk,
    BaseMessage,
)
from langchain_core.runnables import RunnableConfig, RunnableLambda
from langchain_core.tools import tool
from langgraph.graph import END, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt.tool_node import ToolNode
from mlflow.pyfunc import ResponsesAgent
from mlflow.types.responses import (
    ResponsesAgentRequest,
    ResponsesAgentResponse,
    ResponsesAgentStreamEvent,
    output_to_responses_items_stream,
    to_chat_completions_input,
)

logger = logging.getLogger(__name__)
logging.basicConfig(level=os.getenv("LOG_LEVEL", "INFO"))


############################################
# LLMエンドポイントとシステムプロンプトを定義
############################################
# TODO: モデルサービングエンドポイントを置き換えてください
LLM_ENDPOINT_NAME = "databricks-claude-3-7-sonnet"

# TODO: システムプロンプトを更新してください
SYSTEM_PROMPT = "あなたは役に立つアシスタントです。利用可能なツールを使用して質問に答えてください。"

# TODO: エージェントが使用するlakebaseインスタンスの値を入力してください
LAKEBASE_INSTANCE_NAME = "lakebase-name"

# TODO: セマンティックメモリ検索に必要な埋め込み設定値を更新してください
# テキスト埋め込み用のモデルサービングエンドポイントの例 https://docs.databricks.com/aws/en/machine-learning/foundation-model-apis/supported-models#gte-large-en
EMBEDDING_ENDPOINT = "databricks-gte-large-en"  
EMBEDDING_DIMS = 1024

###############################################################################
## エージェント用のツールを定義し、テキスト生成以外のデータ取得やアクションを
## 実行できるようにします
## さらにツールを作成し、使用例を確認するには、以下を参照してください
## https://docs.databricks.com/en/generative-ai/agent-framework/agent-tool.html
###############################################################################

tools = []

# UCツールの例。必要に応じて追加してください
UC_TOOL_NAMES: list[str] = []
if UC_TOOL_NAMES:
    uc_toolkit = UCFunctionToolkit(function_names=UC_TOOL_NAMES)
    tools.extend(uc_toolkit.tools)

# Databricksベクトル検索インデックスをツールとして使用
# https://docs.databricks.com/en/generative-ai/agent-framework/unstructured-retrieval-tools.html#locally-develop-vector-search-retriever-tools-with-ai-bridge を参照
# 非構造化検索用のベクトル検索ツールインスタンスを格納するリスト
VECTOR_SEARCH_TOOLS = []

# ベクトル検索リトリーバーツールを追加するには、
# VectorSearchRetrieverToolとcreate_tool_infoを使用し、
# 結果をTOOL_INFOSに追加します。
# 例:
# VECTOR_SEARCH_TOOLS.append(
#     VectorSearchRetrieverTool(
#         index_name="",
#         # filters="..."
#     )
# )

tools.extend(VECTOR_SEARCH_TOOLS)

#####################
## エージェントロジックを定義
#####################


class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]
    custom_inputs: Optional[dict[str, Any]]
    custom_outputs: Optional[dict[str, Any]]
    user_id: Optional[str]


class LangGraphResponsesAgent(ResponsesAgent):
    """ユーザーベースの長期記憶を持つResponsesAgentを使用したステートレスエージェント。

    機能:
    - DatabricksStoreを介した認証情報ローテーション付きの接続プール
    - ユーザーベースの長期記憶永続化(メモリは"users".user_idの下に"store"テーブルに保存)
    - UC関数を使用したツールサポート
    - 自動接続管理 - スケーラビリティのために操作ごとに接続を借用
    """

    def __init__(self):
        self.lakebase_instance_name = LAKEBASE_INSTANCE_NAME
        self.system_prompt = SYSTEM_PROMPT
        self.model = ChatDatabricks(endpoint=LLM_ENDPOINT_NAME)

        self._store = None
        self._memory_tools = None

    @property
    def store(self):
        """セマンティック検索サポート付きDatabricksStoreの遅延初期化。"""
        if self._store is None:
            logger.info(f"インスタンス: {self.lakebase_instance_name}、埋め込みエンドポイント {EMBEDDING_ENDPOINT}、次元 {EMBEDDING_DIMS} でDatabricksStoreを初期化中")
            self._store = DatabricksStore(
                instance_name=self.lakebase_instance_name,
                embedding_endpoint=EMBEDDING_ENDPOINT,
                embedding_dims=EMBEDDING_DIMS,
            )
            self._store.setup()
        return self._store

    @property
    def memory_tools(self):
        """メモリツールの遅延初期化。"""
        if self._memory_tools is None:
            logger.info("メモリツールを作成中")
            self._memory_tools = self._create_memory_tools()
        return self._memory_tools

    @property
    def model_with_all_tools(self):
        all_tools = tools + self.memory_tools
        return self.model.bind_tools(all_tools) if all_tools else self.model

    def _create_memory_tools(self):
        """長期記憶の読み書き用ツールを作成。"""

        @tool
        def get_user_memory(query: str, config: RunnableConfig) -> str:
            """ベクトル埋め込みを介したセマンティック検索を使用して、長期記憶からユーザーに関する関連情報を検索します。

            このツールを使用して、ユーザーの好み、共有された事実、その他の個人情報など、
            以前に保存されたユーザーに関する情報を取得します。

            Args:
            """
            user_id = config.get("configurable", {}).get("user_id")
            if not user_id:
                return "メモリは利用できません - user_idが提供されていません。"

            namespace = ("user_memories", user_id.replace(".", "-"))

            results = self.store.search(namespace, query=query, limit=5)

            if not results:
                return "このユーザーのメモリが見つかりませんでした。"

            memory_items = []
            for item in results:
                memory_items.append(f"- [{item.key}]: {json.dumps(item.value)}")

            return f"{len(results)}件の関連メモリが見つかりました(セマンティック類似度でランク付け):\n" + "\n".join(memory_items)

        @tool
        def save_user_memory(memory_key: str, memory_data_json: str, config: RunnableConfig) -> str:
            """ベクトル埋め込みを使用してユーザーに関する情報を長期記憶に保存します。

            このツールを使用して、ユーザーが共有した重要な情報、
            例えば好み、事実、その他の個人情報を記憶します。

            Args:
                memory_key: このメモリの説明的なキー(例: "preferences", "favorite_color", "background_info")
                memory_data_json: 記憶する情報を含むJSON文字列。
                    例: '{"favorite_color": "purple"}'
            """
            user_id = config.get("configurable", {}).get("user_id")
            if not user_id:
                return "メモリを保存できません - user_idが提供されていません。"

            namespace = ("user_memories", user_id.replace(".", "-"))

            try:
                memory_data = json.loads(memory_data_json)
                # memory_dataが辞書であることを検証(リストやその他の型ではない)
                if not isinstance(memory_data, dict):
                    return f"メモリの保存に失敗しました: memory_dataはJSONオブジェクト(辞書)である必要があります。{type(memory_data).__name__}ではありません。例: '{{\"key\": \"value\"}}'"
                self.store.put(namespace, memory_key, memory_data)
                return f"キー '{memory_key}' でユーザーのメモリを正常に保存しました。"
            except json.JSONDecodeError as e:
                return f"メモリの保存に失敗しました: 無効なJSON形式 - {str(e)}"

        @tool
        def delete_user_memory(memory_key: str, config: RunnableConfig) -> str:
            """ユーザーの長期記憶から特定のメモリを削除します。

            ユーザーが何かを忘れるように依頼したり、メモリから
            情報を削除するように依頼した場合に、このツールを使用します。

            Args:
                memory_key: 削除するメモリのキー(例: "preferences", "likes", "background_info")
            """
            user_id = config.get("configurable", {}).get("user_id")
            if not user_id:
                return "メモリを削除できません - user_idが提供されていません。"

            namespace = ("user_memories", user_id.replace(".", "-"))

            self.store.delete(namespace, memory_key)
            return f"キー '{memory_key}' でユーザーのメモリを正常に削除しました。"

        return [get_user_memory, save_user_memory, delete_user_memory]

    def _create_graph(self):
        """　LangGraphワークフローを作成"""
        def should_continue(state: AgentState):
            messages = state["messages"]
            last_message = messages[-1]
            if isinstance(last_message, AIMessage) and last_message.tool_calls:
                return "continue"
            return "end"

        model_with_tools = self.model_with_all_tools

        if self.system_prompt:
            preprocessor = RunnableLambda(
                lambda state: [{"role": "system", "content": self.system_prompt}] + state["messages"]
            )
        else:
            preprocessor = RunnableLambda(lambda state: state["messages"])

        model_runnable = preprocessor | model_with_tools

        def call_model(state: AgentState, config: RunnableConfig):
            response = model_runnable.invoke(state, config)
            return {"messages": [response]}

        workflow = StateGraph(AgentState)
        workflow.add_node("agent", RunnableLambda(call_model))

        active_tools = (tools + self.memory_tools)

        if active_tools:
            workflow.add_node("tools", ToolNode(active_tools))
            workflow.add_conditional_edges(
                "agent",
                should_continue,
                {"continue": "tools", "end": END}
            )
            workflow.add_edge("tools", "agent")
        else:
            workflow.add_edge("agent", END)

        workflow.set_entry_point("agent")

        return workflow.compile()

    def _get_user_id(self, request: ResponsesAgentRequest) -> Optional[str]:
        """
        利用可能な場合はチャットコンテキストからuser_idを使用し、提供されていない場合はNoneを返す
        """
        # メモリを保存するuser_idとしてチャットコンテキストからのuser_idを使用
        # https://mlflow.org/docs/latest/api_reference/python_api/mlflow.types.html#mlflow.types.agent.ChatContext
        if request.context and getattr(request.context, "user_id", None):
            return request.context.user_id
        return None

    def predict(self, request: ResponsesAgentRequest) -> ResponsesAgentResponse:
        """非ストリーミング予測"""
        outputs = [
            event.item
            for event in self.predict_stream(request)
            if event.type == "response.output_item.done"
        ]
        return ResponsesAgentResponse(output=outputs)

    def predict_stream(
        self,
        request: ResponsesAgentRequest,
    ) -> Generator[ResponsesAgentStreamEvent, None, None]:
        """ストリーミング予測"""
        user_id = self._get_user_id(request)

        # user_idがない場合、メモリを取得できません
        if not user_id:
            logger.error(
                "user_idがないとメモリを保存または取得できません。"
            )

        ci = dict(request.custom_inputs or {})
        if user_id:
            ci["user_id"] = user_id
        request.custom_inputs = ci

        cc_msgs = to_chat_completions_input([i.model_dump() for i in request.input])

        run_config = {"configurable": {}}
        if user_id:
            run_config["configurable"]["user_id"] = user_id

        graph = self._create_graph()

        state_input = {"messages": cc_msgs}
        if user_id:
            state_input["user_id"] = user_id

        # グラフ実行をストリーミング
        for event in graph.stream(
            state_input,
            run_config,
            stream_mode=["updates", "messages"]
        ):
            if event[0] == "updates":
                for node_data in event[1].values():
                    if len(node_data.get("messages", [])) > 0:
                        yield from output_to_responses_items_stream(node_data["messages"])
            # リアルタイムテキスト生成のためにメッセージチャンクをストリーミング
            elif event[0] == "messages":
                try:
                    chunk = event[1][0]
                    if isinstance(chunk, AIMessageChunk) and (content := chunk.content):
                        yield ResponsesAgentStreamEvent(
                            **self.create_text_delta(delta=content, item_id=chunk.id),
                        )
                except Exception as e:
                    logger.error(f"チャンクのストリーミングエラー: {e}")

# ----- モデルをエクスポート -----
mlflow.langchain.autolog()
AGENT = LangGraphResponsesAgent()
mlflow.models.set_model(AGENT)

# エージェントをローカルでテスト

In [0]:
dbutils.library.restartPython()

In [0]:
# ChatContextからのuser_idを入力user_idとして使用する例
# https://mlflow.org/docs/latest/api_reference/python_api/mlflow.types.html#mlflow.types.agent.ChatContext
from agent import AGENT
import mlflow
from mlflow.types.responses import (
    ResponsesAgentRequest,
    ChatContext
)

req = ResponsesAgentRequest(
    input=[{"role": "user", "content": "Please remember I use Databricks and I am a python developer who likes pistachios and has a dog named Fluffy"}],
    context=ChatContext(
        conversation_id="abc",
        user_id="email@databricks.com"
    ),
)
result = AGENT.predict(req)

print(result.model_dump(exclude_none=True))

In [0]:
# メモリ呼び出しの例

req = ResponsesAgentRequest(
    input=[{"role": "user", "content": "What data platform do I use?"}],
    context=ChatContext(
        conversation_id="abc",
        user_id="email@databricks.com"
    ),
)
result = AGENT.predict(req)

print(result.model_dump(exclude_none=True))

# エージェントをMLflowモデルとしてログ記録
agent.pyファイルからエージェントをコードとしてログ記録します。[MLflow - Models from Code](https://mlflow.org/docs/latest/models.html#models-from-code)を参照してください。

## Databricksリソースの自動認証を有効化
最も一般的なDatabricksリソースタイプについて、Databricksはログ記録時にエージェントのリソース依存関係を事前に宣言することをサポートし、推奨しています。これにより、エージェントをデプロイする際に自動認証パススルーが有効になります。自動認証パススルーを使用すると、Databricksはエージェントエンドポイント内からこれらのリソース依存関係に安全にアクセスするための短命認証情報を自動的にプロビジョニング、ローテーション、管理します。

自動認証を有効にするには、`mlflow.pyfunc.log_model()`を呼び出す際に依存するDatabricksリソースを指定します。

**TODO:** 
- lakebaseをリソースタイプとして追加
- Unity Catalogツールが[ベクトル検索インデックス](https://docs.databricks.com/docs%20link)をクエリするか、[外部関数](https://docs.databricks.com/docs%20link)を利用する場合、依存するベクトル検索インデックスとUC接続オブジェクトをそれぞれリソースとして含める必要があります。ドキュメントを参照してください([AWS](https://docs.databricks.com/generative-ai/agent-framework/log-agent.html#specify-resources-for-automatic-authentication-passthrough) | [Azure](https://learn.microsoft.com/azure/databricks/generative-ai/agent-framework/log-agent#resources))。

In [0]:
# デプロイ時に自動認証パススルー用に指定するDatabricksリソースを決定
import mlflow
from agent import tools, LLM_ENDPOINT_NAME, LAKEBASE_INSTANCE_NAME
from databricks_langchain import VectorSearchRetrieverTool
from mlflow.models.resources import DatabricksFunction, DatabricksServingEndpoint, DatabricksLakebase
from unitycatalog.ai.langchain.toolkit import UnityCatalogTool
from pkg_resources import get_distribution

resources = [DatabricksServingEndpoint(LLM_ENDPOINT_NAME), DatabricksLakebase(database_instance_name=LAKEBASE_INSTANCE_NAME)]

for tool in tools:
    if isinstance(tool, VectorSearchRetrieverTool):
        resources.extend(tool.resources)
    elif isinstance(tool, UnityCatalogTool):
        resources.append(DatabricksFunction(function_name=tool.uc_function_name))

input_example = {
    "input": [
        {
            "role": "user",
            "content": "What is an LLM agent?"
        }
    ],
}

logged_agent_info = mlflow.pyfunc.log_model(
    name="agent",
    python_model="agent.py",
    input_example=input_example,
    resources=resources,
    pip_requirements=[
        "mlflow==3.6.0",
        f"databricks-langchain[memory]=={get_distribution('databricks-langchain[memory]').version}",
    ]
)

# Agent Evaluationでエージェントを評価
Mosaic AI Agent Evaluationを使用して、期待される応答やその他の評価基準に基づいてエージェントの応答を評価します。指定した評価基準を使用して反復をガイドし、MLflowを使用して計算された品質メトリクスを追跡します。Databricksドキュメントを参照してください([AWS](https://docs.databricks.com/(https://docs.databricks.com/aws/generative-ai/agent-evaluation) | [Azure](https://learn.microsoft.com/azure/databricks/generative-ai/agent-evaluation/))。

ツール呼び出しを評価するには、カスタムメトリクスを追加します。Databricksドキュメントを参照してください([AWS](https://docs.databricks.com/en/generative-ai/agent-evaluation/custom-metrics.html#evaluating-tool-calls) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/generative-ai/agent-evaluation/custom-metrics#evaluating-tool-calls))。

In [0]:
import mlflow
from mlflow.genai.scorers import RelevanceToQuery, RetrievalGroundedness, RetrievalRelevance, Safety

eval_dataset = [
    {
        "inputs": {"input": [{"role": "user", "content": "Calculate the 15th Fibonacci number"}]},
        "expected_response": "The 15th Fibonacci number is 610.",
    }
]

eval_results = mlflow.genai.evaluate(
    data=eval_dataset,
    predict_fn=lambda input: AGENT.predict({"input": input}),
    scorers=[RelevanceToQuery(), Safety()],  # 該当する場合はさらにスコアラーを追加
)

# MLflow UIで評価結果を確認(コンソール出力を参照)

# デプロイ前のエージェント検証
エージェントを登録およびデプロイする前に、mlflow.models.predict() APIを使用してデプロイ前チェックを実行します。

In [0]:
mlflow.models.predict(
    model_uri=logged_agent_info.model_uri,
    input_data={"input": [{"role": "user", "content": "I am working on stateful agents"}]},
    env_manager="uv",
)

# モデルをUnity Catalogに登録
以下の`catalog`、`schema`、`model_name`を更新して、MLflowモデルをUnity Catalogに登録します。

In [0]:
mlflow.set_registry_uri("databricks-uc")

# TODO: UCモデルのカタログ、スキーマ、モデル名を定義
catalog = "catalog"
schema = "schema"
model_name = "long-term-memory-agent"

UC_MODEL_NAME = f"{catalog}.{schema}.{model_name}"

# モデルをUCに登録
uc_registered_model_info = mlflow.register_model(
    model_uri=logged_agent_info.model_uri, name=UC_MODEL_NAME
)

エージェントをデプロイ

In [0]:
from databricks import agents
agents.deploy(UC_MODEL_NAME, uc_registered_model_info.version, tags = {"endpointSource": "docs"})

# 次のステップ
エージェントのデプロイが完了するまで約15分かかります。エージェントがデプロイされた後、AI playgroundでチャットして追加のチェックを実行したり、組織内のSMEと共有してフィードバックを得たり、本番アプリケーションに組み込んだりできます。

これで、ステートフルエージェントを使用して、過去のスレッドを取得して会話を続けることができます。

Lakebaseインスタンスをクエリして、ユーザーメモリのレコードを確認できます。以下はストア内のアイテムを確認する基本的なクエリです:
```
select *
from public.store
order by updated_at desc
limit 50;
```